# QLoRA 低资源微调实战

## QLoRA = Quantization + LoRA

QLoRA（Quantized LoRA）是 Dettmers et al. (2023) 提出的方法，核心思想：

1. **4-bit 量化**：将预训练模型的权重从 FP16（16位）压缩到 NF4（4位），显存占用降低 ~4 倍
2. **LoRA 微调**：在量化后的模型上应用 LoRA，只训练少量低秩参数
3. **双重量化**：对量化常数再次量化，进一步节省显存

### 为什么 QLoRA 重要？

```
7B 模型全量微调 (FP32):  ~28 GB 显存  → 需要 A100 80GB
7B 模型 LoRA (FP16):     ~14 GB 显存  → 需要 V100 32GB
7B 模型 QLoRA (4-bit):   ~4-6 GB 显存 → RTX 4060 即可！🎉
```

**B站场景**：团队预算有限，用消费级显卡（RTX 4060/4070）就能微调 7B 模型生成广告文案！

### NF4（NormalFloat4）量化

NF4 不是简单的均匀量化，而是针对**正态分布**的权重设计的最优4位数据类型：
- 预训练权重通常服从正态分布
- NF4 在正态分布的高密度区域分配更多量化级别
- 比普通 INT4 量化精度更高

In [ ]:
"""BitsAndBytesConfig 4-bit 量化配置"""

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

    # ============================================================
    # QLoRA 配置：4-bit 量化 + LoRA
    # ============================================================

    # Step 1: 配置 4-bit 量化
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,                    # 启用 4-bit 量化
        bnb_4bit_quant_type="nf4",            # NF4 量化类型（推荐）
        bnb_4bit_compute_dtype=torch.bfloat16, # 计算时用 BF16（精度更好）
        bnb_4bit_use_double_quant=True,        # 双重量化（进一步压缩）
    )

    # Step 2: 加载量化后的模型
    model_name = "Qwen/Qwen2-7B"
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",         # 自动分配到可用 GPU
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # Step 3: 为 k-bit 训练做准备
    model = prepare_model_for_kbit_training(model)

    # Step 4: 配置 LoRA
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none",
    )

    # Step 5: 应用 LoRA
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    print("\n✅ QLoRA 模型准备完毕！")
    print(f"模型显存占用：{model.get_memory_footprint() / 1e9:.2f} GB")

except ImportError as e:
    print(f"⚠️ 缺少依赖包：{e}")
    print("请安装：pip install transformers peft bitsandbytes torch")
    print("\n模拟 QLoRA 配置输出：")
    print("---")
    print("BitsAndBytesConfig(")
    print("  load_in_4bit=True,              # 4-bit量化")
    print("  bnb_4bit_quant_type='nf4',      # NF4量化")
    print("  bnb_4bit_compute_dtype=bfloat16, # BF16计算")
    print("  bnb_4bit_use_double_quant=True   # 双重量化")
    print(")")
    print("\n预期效果（Qwen2-7B + QLoRA）：")
    print("  模型显存占用: ~4.5 GB (原FP32需要 ~28 GB)")
    print("  可训练参数: ~8M (0.1%)")
    print("  适合显卡: RTX 4060 (8GB) 及以上")

In [ ]:
"""不同精度下的显存占用对比"""
import numpy as np

# ============================================================
# 显存占用对比：FP32 vs FP16 vs INT8 vs INT4
# ============================================================

model_sizes = {
    "Qwen2-1.5B": 1.5e9,
    "Qwen2-7B": 7e9,
    "Llama3-8B": 8e9,
    "Qwen2-14B": 14e9,
    "Llama3-70B": 70e9,
}

# 每个参数占用的字节数
precision_bytes = {
    "FP32": 4,
    "FP16/BF16": 2,
    "INT8": 1,
    "INT4(QLoRA)": 0.5,
}

print("=" * 75)
print("不同精度下的模型显存占用（仅权重，不含梯度和优化器状态）")
print("=" * 75)
print(f"{'模型':<15}", end="")
for prec in precision_bytes:
    print(f"{prec:>14}", end="")
print()
print("-" * 75)

for model_name, params in model_sizes.items():
    print(f"{model_name:<15}", end="")
    for prec_name, bytes_per_param in precision_bytes.items():
        size_gb = params * bytes_per_param / 1e9
        print(f"{size_gb:>11.1f} GB", end="")
    print()

# B站实际场景分析
print("\n" + "=" * 75)
print("B站广告文案微调场景 GPU 选择建议")
print("=" * 75)

gpu_options = [
    ("RTX 4060", 8, "INT4", "Qwen2-7B + QLoRA"),
    ("RTX 4070 Ti", 12, "INT4", "Qwen2-7B + QLoRA（更舒适）"),
    ("RTX 4090", 24, "FP16", "Qwen2-7B + LoRA"),
    ("A100 40GB", 40, "FP16", "Qwen2-14B + LoRA"),
    ("A100 80GB", 80, "FP16", "Llama3-70B + QLoRA"),
]

print(f"{'GPU':<18} {'显存':>8} {'精度':>8} {'推荐方案'}")
print("-" * 65)
for gpu, vram, prec, plan in gpu_options:
    print(f"{gpu:<18} {vram:>6} GB {prec:>8}   {plan}")

print("\n💡 结论：QLoRA 让消费级显卡也能微调大模型，极大降低了B站团队的硬件门槛")

## QLoRA 实战技巧

### 什么时候用 QLoRA vs 全精度 LoRA？

| 场景 | 推荐方案 | 原因 |
|------|----------|------|
| GPU 显存 < 16GB | QLoRA | 唯一选择，否则放不下 |
| GPU 显存 16-24GB | QLoRA 或 LoRA | 都可以，QLoRA更省显存可开更大batch |
| GPU 显存 > 40GB | LoRA (FP16) | 精度更好，训练更稳定 |
| 追求极致效果 | LoRA (FP16) | 量化会有少量精度损失 |
| 快速实验/原型 | QLoRA | 省钱省时间 |

### QLoRA 常见问题及解决

1. **训练不稳定/loss跳动**
   - 降低学习率（1e-5 → 5e-6）
   - 使用 `bnb_4bit_compute_dtype=bfloat16`（比 float16 更稳定）

2. **效果比 LoRA 差**
   - 适当增大 r（如 16 → 32）
   - 增加训练轮数
   - 检查量化后模型的基线效果

3. **bitsandbytes 安装失败**
   - 确保 CUDA 版本兼容
   - Linux: `pip install bitsandbytes`
   - Windows: `pip install bitsandbytes-windows`

## 面试要点 🎯

### 1. QLoRA 的三大核心技术是什么？

> 1. **NF4 量化**：针对正态分布权重优化的 4-bit 数据类型，比普通 INT4 精度更高
> 2. **双重量化（Double Quantization）**：对量化常数再次量化，每个参数额外节省 ~0.37 bit
> 3. **分页优化器（Paged Optimizers）**：利用 NVIDIA 统一内存，在 GPU 显存不足时自动 offload 到 CPU

### 2. 4-bit 量化会损失多少精度？

> - NF4 + 双重量化的精度损失非常小，论文实验表明 QLoRA 在多数任务上与 16-bit LoRA 效果相当
> - 关键是使用 NF4 而非普通 INT4，以及计算时反量化到 BF16
> - B站广告文案场景（生成任务）：QLoRA 效果完全够用

### 3. QLoRA 训练时的显存构成是怎样的？

> ```
> 总显存 = 4-bit模型权重 + LoRA参数(FP16) + 梯度(FP16) + 优化器状态(FP32) + 激活值
>        = ~3.5 GB        + ~8 MB           + ~8 MB     + ~16 MB           + ~1-2 GB
>        ≈ 5-6 GB（7B模型，batch_size=1）
> ```

### 4. 实际项目中 QLoRA 的工作流？

> 1. 选择基座模型（如 Qwen2-7B）
> 2. 准备B站广告文案的 SFT 数据（instruction/output 格式）
> 3. 用 QLoRA 在消费级 GPU 上训练
> 4. 合并 adapter 到基座模型（`model.merge_and_unload()`）
> 5. 量化部署（GPTQ/AWQ）到生产环境